# Silver Layer: Sales Details CDC (Free Edition)

**Pattern**: CDC via `foreachBatch` Merge  
**Trigger**: AvailableNow (Incremental Batch)

In [0]:
from pyspark.sql.functions import col
from delta.tables import DeltaTable

bronze_table = "workspace.bronze.crm_sales_details_cdc"
silver_table = "workspace.silver.crm_sales_details_cdc"
checkpoint_path = "/Volumes/workspace/bronze/raw_sources/checkpoints/silver_crm_sales_details"

def upsert_sales(batch_df, batch_id):
    transformed_df = batch_df.select(
        col("sls_ord_num").alias("order_number"),
        col("sls_prd_key").alias("product_key"),
        col("sls_cust_id").alias("customer_id"),
        col("sls_order_dt").alias("order_date"),
        col("sls_ship_dt").alias("ship_date"),
        col("sls_due_dt").alias("due_date"),
        col("sls_sales").alias("sales_amount"),
        col("sls_quantity").alias("quantity"),
        col("sls_price").alias("price"),
        col("ingestion_timestamp")
    ).filter(col("order_number").isNotNull() & col("product_key").isNotNull())

    if not spark.catalog.tableExists(silver_table):
        transformed_df.write.format("delta").saveAsTable(silver_table)
        return

    target_table = DeltaTable.forName(spark, silver_table)
    (target_table.alias("t")
        .merge(transformed_df.alias("s"), "t.order_number = s.order_number AND t.product_key = s.product_key")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

print(f"Starting incremental update for {silver_table}...")
query = (
    spark.readStream
    .table(bronze_table)
    .writeStream
    .foreachBatch(upsert_sales)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
print(f"✓ Silver update complete. Total records in {silver_table}: {spark.table(silver_table).count():,}")